# Assignment Overview

In this assignment we will be labeling patient images for Fitzpatrick types and creating segmentation masks of dermoscopy images. We then dertermine the majority concensus from the different labels and evaluate our own labels against this majority concensus using metrics. While identifying the majority concensus, we keep looking for any disagreements or ambiguities.

# Import Libraries and load data

In [2]:
import os
import glob

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import json

from PIL import Image
from sklearn.metrics import precision_score, recall_score, f1_score

In [ ]:
import plotly.io as pio
pio.renderers.default = 'notebook_connected'

In [3]:
DATA_DIR1 = 'labels_fitzpatrick30'
DATA_DIR2 = 'labels_isic10'
LABEL_NAMES1 = ['I', 'II', 'III', 'IV', 'V', 'VI']
LABEL_NUMBER1 = [1, 2, 3, 4, 5, 6]
LABEL_NAMES2 = ['lesion', 'non-lesion']

# `Task 1: Label 30 patient images for the Fitzpatrick type.`

The Fitzpatrick scale is numerically based from I-VI, with I being the lightes color skin type and VI the darkest.

[![Image Alt Text](https://barrislaser.com/wp-content/uploads/2020/03/fitzpatrick-skin-type-scale_219892-800x294.png)]

Source: (https://barrislaser.com/wp-content/uploads/2020/03/fitzpatrick-skin-type-scale_219892-800x294.png)

### Remarks for Task 1
- The lightning of the flash from the camera reflects from the skin, which is why my decission on the skin type was more focused on the border of the image. 
- The first 1-10 image was analyzed more precise than at the end of the 30 patient images, because it became easier to identify the skin type (repetetivness).
- Different diseases on the skin was also a factor to consider, because it was not always clear if the skin type was influenced by the disease or not.

# `Task 2: Create segmentation masks for 10 dermoscopy images of melanocytic lesions.`


| ISIC10 Images                  | Segmentation                |
|-----------------------------------|-----------------------------|
| <img src='labels_isic10/data_isic10/ISIC_0000000.jpg' alt='1 isic image' width='200' height='150'> | <img src='labels_isic10/choekyel_isic10/ISIC_0000000_Segmentation.png' alt='1 isic image segmentation' width='200' height="150">|
| <img src="labels_isic10/data_isic10/ISIC_0000001.jpg" alt='2 isic image' width='200' height="150"> | <img src="labels_isic10/choekyel_isic10/ISIC_0000001_Segmentation.png" alt="2 isic image segmentation" width="200" height="150">|
| <img src="labels_isic10/data_isic10/ISIC_0000002.jpg" alt="3 isic image" width="200" height="150"> | <img src="labels_isic10/choekyel_isic10/ISIC_0000002_Segmentation.png" alt="3 isic image segmentation" width="200" height="150">|
| <img src="labels_isic10/data_isic10/ISIC_0000004.jpg" alt="5 isic image" width="200" height="150"> | <img src="labels_isic10/choekyel_isic10/ISIC_0000004_Segmentation.png" alt="5 isic image segmentation" width="200" height="150">|
| <img src="labels_isic10/data_isic10/ISIC_0000006.jpg" alt="7 isic image" width="200" height="150"> | <img src="labels_isic10/choekyel_isic10/ISIC_0000006_Segmentation.png" alt="7 isic image segmentation" width="200" height="150">|
| <img src="labels_isic10/data_isic10/ISIC_0000007.jpg" alt="8 isic image" width="200" height="150"> | <img src="labels_isic10/choekyel_isic10/ISIC_0000007_Segmentation.png" alt="8 isic image segmentation" width="200" height="150">|
| <img src="labels_isic10/data_isic10/ISIC_0000008.jpg" alt="9 isic image" width="200" height="150"> | <img src="labels_isic10/choekyel_isic10/ISIC_0000008_Segmentation.png" alt="9 isic image segmentation" width="200" height="150">|
| <img src="labels_isic10/data_isic10/ISIC_0000009.jpg" alt="10 isic image" width="200" height="150">| <img src="labels_isic10/choekyel_isic10/ISIC_0000009_Segmentation.png" alt="10 isic image segmentation" width="200" height="150">|
| <img src="labels_isic10/data_isic10/ISIC_0000011.jpg" alt="11 isic image" width="200" height="150"> | <img src="labels_isic10/choekyel_isic10/ISIC_0000011_Segmentation.png" alt="11 isic image segmentation" width="200" height="150">|

### Remarks for Task 2
- The segmentation was straightforward, because the melanocytic lesions were clearly visible. Only the 4th segmentation was a bit difficult, because the lesion was not clearly visible or rather i was not sure if the lesion was the dark spot or the red spot. I decided to label the red spot as the lesion.

# `Task 3: Determine AMLMED majority consensus.`

I will first load the data, select the columns needed, fill any missing values and then determine the majority consensus.

In [4]:
def load_data(csv_file, id_column, important_column):
    df = pd.read_csv(csv_file)
    labels = df[[id_column, important_column]]
    return labels

def fill_nan_with_mode(row):
    mode = row.mode().iloc[0] 
    return row.fillna(mode) 

def calculate_majority(row):
    counts = row.drop('choice').value_counts()
    majority_choice = None
    
    for label in LABEL_NUMBER1:
        if label in counts:
            if majority_choice is None or counts[label] > counts[majority_choice]:
                majority_choice = label
    return majority_choice

In [5]:
df_fitzpatrick = pd.DataFrame(index=range(1, 31)) 
df_fitzpatrick.index.name = 'annotation_id'

label_to_number = {'I': 1, 'II': 2, 'III': 3, 'IV': 4, 'V': 5, 'VI': 6} 
counter = 1
for filename in os.listdir(DATA_DIR1):
    if filename.endswith('.csv'):
        student_csv_file = os.path.join(DATA_DIR1, filename)
        student_df = load_data(student_csv_file, 'annotation_id', 'choice')
        df_fitzpatrick = df_fitzpatrick.join(student_df.set_index('annotation_id'), rsuffix=f'_stud{counter}') 
        counter += 1

df_fitzpatrick = df_fitzpatrick.apply(fill_nan_with_mode, axis=1)

for column in df_fitzpatrick.columns:
    df_fitzpatrick[column] = df_fitzpatrick[column].map(label_to_number)

df_fitzpatrick['majority_consensus'] = df_fitzpatrick.apply(calculate_majority, axis=1)
df_fitzpatrick.reset_index(inplace=True)
df_fitzpatrick

,annotation_id,choice,choice_stud2,choice_stud3,choice_stud4,choice_stud5,choice_stud6,choice_stud7,choice_stud8,choice_stud9,majority_consensus
0,1,6,5,5,4,5,5,5,5,5,5
1,2,1,4,1,1,3,2,2,1,2,1
2,3,2,2,2,2,3,2,2,1,1,2
3,4,4,5,3,3,4,4,4,2,4,4
4,5,5,4,3,3,5,3,4,3,4,3
5,6,6,6,6,5,6,6,4,5,6,6
6,7,5,4,4,3,5,4,4,3,5,4
7,8,5,6,6,5,6,5,5,6,5,5
8,9,6,6,6,6,6,6,6,6,6,6
9,10,5,5,6,5,6,6,6,5,6,6


I have a dataframe of alle the AMLMED choices and the respective majority consensus for each image.

Next I will be creating a new dataframe where i stack the AMLMED choices on top if chosen. This will allow me to see the different choices for each image. I will then be able to see if there are any disagreements or ambiguities.

In [6]:
selected_columns = df_fitzpatrick.iloc[:, 2:10] # 'annotation_id', 'choice' and 'majority_consensus' is not selected
label_counts = {label: [] for label in LABEL_NUMBER1}

for index, row in df_fitzpatrick.iterrows():
    counts = row[selected_columns.columns].value_counts().to_dict()
    for label in LABEL_NUMBER1:
        label_counts[label].append(counts.get(label, 0))

label_counts_df = pd.DataFrame(label_counts)
label_counts_df.index = df_fitzpatrick.index

In [7]:
label_counts_df

,1,2,3,4,5,6
0,0,0,0,1,7,0
1,3,3,1,1,0,0
2,2,5,1,0,0,0
3,0,1,2,4,1,0
4,0,0,4,3,1,0
5,0,0,0,1,2,5
6,0,0,2,4,2,0
7,0,0,0,0,4,4
8,0,0,0,0,0,8
9,0,0,0,0,3,5


In [8]:
majority_consensus_values = df_fitzpatrick['majority_consensus']
my_choice_values = df_fitzpatrick['choice']

heatmap_data = label_counts_df.T

fig_heatmap = go.Figure(data=go.Heatmap(
        z=heatmap_data.values,
        x=heatmap_data.columns,
        y=LABEL_NAMES1,
        colorscale='YlGnBu',
        colorbar=dict(
            title='Classification',
            tickvals=[0, 1, 2, 3, 4, 5, 6, 7],
            ticktext=["No Classification", "1", "2", "3", "4", "5", "6", "High Classification"]
        )
    ))

fig_heatmap.update_layout(
    xaxis=dict(title='Annotation ID', ticktext=df_fitzpatrick['annotation_id']),
    yaxis=dict(title='Labels', showgrid=False),  # Hide y-axis grid lines
    title='Heatmap of Label Distribution and Disagreement Among Students',
)

fig_heatmap.update_traces(
    hovertemplate='Annotation ID: %{x}<br>Label: %{y}<br>Frequency: %{z}',
)

fig_heatmap.update_xaxes(tickvals=list(heatmap_data.columns))

for annotation_id, consensus_label in enumerate(majority_consensus_values):
    fig_heatmap.add_shape(
        type='rect',
        x0=annotation_id - 0.5,  
        x1=annotation_id + 0.5,  
        y0=(consensus_label- 1) - 0.5, 
        y1=(consensus_label - 1) + 0.5,  
        line=dict(color='red', width=2),
        fillcolor='rgba(0,0,0,0)', 
    )
for annotation_id, consensus_label in enumerate(my_choice_values):
    fig_heatmap.add_shape(
        type='line', 
        x0=annotation_id - 0.5,  
        x1=annotation_id + 0.5,  
        y0=(consensus_label- 1) - 0.5,  
        y1=(consensus_label - 1) + 0.5, 
        line=dict(color='red', width=2, dash='dot'),
    )
    fig_heatmap.add_shape(
        type='line',  
        x0=annotation_id - 0.5, 
        x1=annotation_id + 0.5,  
        y0=(consensus_label- 1) + 0.5,  
        y1=(consensus_label - 1) - 0.5,
        line=dict(color='red', width=2, dash='dot'),
    )

fig_heatmap.show()

The Heatmap shows the label distribution and diagreement between the different raters. The darker the color, the more they agree. The red box <font color=#FF0000>**[]**</font> indicates the majority consensus and the red dotted cross <font color=#FF0000>**X**</font> indicates my own labeling. 

| Fitzpatrick30 Images                  | Findings from Heatmap                |
|-----------------------------------|-----------------------------|
| <img src="labels_fitzpatrick30/data_fitzpatrick30/3901.jpg" width="200" height="150">  | My labeling for the following image was the worst (Annotation ID = 5). I think because of the big surface and with the reflected lightig the majority thought it was on the lighter side of the skin type.|
| <img src="labels_fitzpatrick30/data_fitzpatrick30/6295.jpg" width="200" height="150">  | The raters of the following image were unsure how to label the skin type (Annotation ID = 15). Could be because the patients condition influenced his skin which led to labeling ambiguities.|
| <img src="labels_fitzpatrick30/data_fitzpatrick30/4099.jpg" width="200" height="150"> <img src="labels_fitzpatrick30/data_fitzpatrick30/8513.jpg" width="200" height="150"> <img src="labels_fitzpatrick30/data_fitzpatrick30/folliculitis83.jpg" width="200" height="150"> <img src="labels_fitzpatrick30/data_fitzpatrick30/juvenile-xanthogranuloma80.jpg" width="200" height="150"> <img src="labels_fitzpatrick30/data_fitzpatrick30/seborrheic-keratoses22.jpg" width="200" height="150"> | For these five images all the raters were in agreement (Annotation ID = 9,17,22,24,29). The similarity between all the images are, they are at the extreme end of the skin type.|

# `Task 4: Evaluate your labels against consensus.`

I computed the precision, recall and F1 score for my labels against the majority consensus.

In [9]:
y_true = df_fitzpatrick['majority_consensus'].values
y_pred = df_fitzpatrick['choice'].values

precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')

print(f" Precision: {precision:.2f}")
print(f" Recall: {recall:.2f}")
print(f" F1 score: {f1:.2f}")


 Precision: 0.67
 Recall: 0.57
 F1 score: 0.57


With the precision score of **0.67**, it means that 67% of the time when i label a skin type as a certain type, it is actually that type. The recall score of **0.57** means that 57% of the time when the majority consensus labels a skin type as a certain type, i label it as that type. The F1 score of **0.57** is the harmonic mean of the precision and recall score. The F1 score is a better measure of a model's performance than the accuracy score, because it takes into account both the precision and recall score. With the 57% F1 score, it means that i have no bias towards minimizing false positives or capturing all positive cases.

Comparing it with the heatmap it makes sense.

In [25]:
from brush import decode_rle, bytes2bit, InputStream, decode_from_annotation

In [62]:
df_isic = pd.DataFrame(index=range(31, 41))
df_isic.index.name = 'annotation_id'

counter = 1
csv_files = glob.glob(os.path.join(DATA_DIR2, '*_isic10/*.csv'))

for student_csv_file in csv_files:
    student_df = load_data(student_csv_file, 'annotation_id', 'tag')
    df_isic = df_isic.join(student_df.set_index('annotation_id'), rsuffix=f'_stud{counter}')
    counter += 1
df_isic.reset_index(inplace=True)

# def extract_rle(tag_str):
#     try:
#         tag_list = json.loads(tag_str)
#         first_item = tag_list[0]
#         rle_values = first_item.get('rle', [])
#         return rle_values
#     except (json.JSONDecodeError, IndexError):
#         return []

# for col in df_isic.columns:
#     if col.startswith('tag'):
#         df_isic[col] = df_isic[col].apply(extract_rle)
#         # df_isic[col] = df_isic[col].apply(decode_rle)

In [63]:
df_isic.head()

,annotation_id,tag,tag_stud2,tag_stud3,tag_stud4,tag_stud5
0,31,"[{""format"":""rle"",""rle"":[0,47,216,8,57,27,255,2...","[{""format"":""rle"",""rle"":[0,47,216,8,57,27,255,2...","[{""format"":""rle"",""rle"":[0,47,216,8,57,27,255,2...","[{""format"":""rle"",""rle"":[0,47,216,8,57,27,255,2...","[{""format"":""rle"",""rle"":[0,47,216,8,57,27,255,2..."
1,32,"[{""format"":""rle"",""rle"":[0,47,216,8,57,27,255,2...","[{""format"":""rle"",""rle"":[0,47,216,8,57,27,255,2...","[{""format"":""rle"",""rle"":[0,47,216,8,57,27,255,2...","[{""format"":""rle"",""rle"":[0,47,216,8,57,27,255,2...","[{""format"":""rle"",""rle"":[0,47,216,8,57,27,255,2..."
2,33,"[{""format"":""rle"",""rle"":[0,47,216,8,57,27,255,2...","[{""format"":""rle"",""rle"":[0,47,216,8,57,27,255,2...","[{""format"":""rle"",""rle"":[0,47,216,8,57,27,255,2...","[{""format"":""rle"",""rle"":[0,47,216,8,57,27,255,2...","[{""format"":""rle"",""rle"":[0,47,216,8,57,27,255,2..."
3,34,"[{""format"":""rle"",""rle"":[0,47,216,8,57,27,255,2...","[{""format"":""rle"",""rle"":[0,47,216,8,57,27,255,2...","[{""format"":""rle"",""rle"":[0,47,216,8,57,27,255,2...","[{""format"":""rle"",""rle"":[0,47,216,8,57,27,255,2...","[{""format"":""rle"",""rle"":[0,47,216,8,57,27,255,2..."
4,35,"[{""format"":""rle"",""rle"":[0,47,216,8,57,27,255,2...","[{""format"":""rle"",""rle"":[0,47,216,8,57,27,255,2...","[{""format"":""rle"",""rle"":[0,47,216,8,57,27,255,2...","[{""format"":""rle"",""rle"":[0,47,216,8,57,27,255,2...","[{""format"":""rle"",""rle"":[0,47,216,8,57,27,255,2..."


In [64]:
d = decode_from_annotation('test', df_isic['tag_stud2'].iloc[0])
d

TypeError: string indices must be integers

I will download the labels from other sudents and identify problematic samples/classes. I created a new column and calculated the majority_consensus for each row by finding the label that appears the most frequently.

In [12]:
df_isic

,annotation_id,tag,tag_stud2,tag_stud3,tag_stud4,tag_stud5
0,31,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,32,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,33,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,34,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,35,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
5,36,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
6,37,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
7,38,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
8,39,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
9,40,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [13]:
original_images_folder = f'{DATA_DIR2}/data_isic10'

# Function to overlay masks on an image and calculate majority consensus
def overlay_and_consensus(annotation_id, original_image_path, rle_columns):
    # Load the original image
    original_image = Image.open(original_image_path)
    consensus_mask = np.zeros(original_image.size[::-1], dtype=np.uint8)

    for col in rle_columns:
        rle_mask = df_isic.loc[df_isic['annotation_id'] == annotation_id, col].values[0]
        # rle_mask = decode_rle(rle_)
        # Print mask information for debugging
        print(f"Mask {col}: Shape={rle_mask.shape}, Max={rle_mask.max()}, Min={rle_mask.min()}")
        overlay = Image.fromarray(rle_mask, mode="L")
        overlay = overlay.resize(original_image.size, Image.BILINEAR)
        overlay_array = np.array(overlay)
        print("Size of rle_mask:", rle_mask.size)
        print("Original image size:", original_image.size)
        print("Target shape for overlay:", (original_image.size[0], original_image.size[1]))
        print("Size of overlay:", overlay.size)
        # Visualize the overlay mask (for debugging)
        # Visualize the overlay on the original image (for debugging)
        overlayed_image = original_image.copy()
        overlayed_image.paste(overlay, (0, 0), overlay)
        plt.imshow(overlayed_image)
        plt.title(f"Overlay Mask {col}")
        plt.show()
        plt.imshow(overlay_array, cmap='gray')
        plt.title(f"Overlay Mask {col}")
        plt.show()

        consensus_mask += overlay_array

    threshold = len(rle_columns) // 2 + 1
    majority_consensus = (consensus_mask >= threshold * 255).astype(np.uint8)

    return majority_consensus

# Create a folder to save the majority consensus masks
output_folder = f'{DATA_DIR2}/output_masks'
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

mapping_dict = {
    31: '00',
    32: '01',
    33: '02',
    34: '04',
    35: '06',
    36: '07',
    37: '08',
    38: '09',
    39: '10',
    40: '11'
}
def map_annotation_id(annotation_id):
    return mapping_dict.get(annotation_id, annotation_id)

for index, row in df_isic.iterrows():
    annotation_id = row["annotation_id"]
    mapped_annotation_id = map_annotation_id(annotation_id)
    original_image_path = os.path.join(original_images_folder, f"ISIC_00000{mapped_annotation_id}.jpg")

    rle_columns = [col for col in df_isic.columns if col != "annotation_id"]

    consensus_mask = overlay_and_consensus(annotation_id, original_image_path, rle_columns)

    output_path = os.path.join(output_folder, f"{mapped_annotation_id}_consensus_mask.jpg")
    Image.fromarray(consensus_mask * 255, mode="L").save(output_path)


Mask tag: Shape=(3135496,), Max=255, Min=0


KeyboardInterrupt: 